# Data Manipulation with Pandas — Extra Exercises (Teacher version)


## 0. Setup

Run this cell first. It rebuilds the original tables and adds the new ones.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 110)

# ---- original tables -------------------------------------------------------
departments = pd.DataFrame({
    "dept_id": [10, 20, 30, 40],
    "dept_name": ["Engineering", "Sales", "Marketing", "HR"],
    "location": ["Paris", "Lyon", "Paris", "Lille"],
})

employees = pd.DataFrame({
    "emp_id": range(1, 11),
    "name": ["Alice", "Bob", "Chloe", "David", "Emma",
             "Farid", "Grace", "Hugo", "Ines", "Jack"],
    "dept_id": [10, 10, 20, 20, 30, 30, 10, 40, 20, 99],   # 99 = no matching dept, on purpose
    "salary": [55000, 48000, 62000, 51000, 47000,
               53000, 59000, 45000, 61000, 50000],
    "hire_date": pd.to_datetime([
        "2019-03-01", "2020-07-15", "2018-01-10", "2021-05-20", "2022-02-11",
        "2019-11-30", "2020-09-01", "2023-01-05", "2017-06-17", "2021-08-08"]),
})

sales_q1 = pd.DataFrame({
    "order_id": range(1, 6), "emp_id": [3, 4, 9, 3, 4],
    "product": ["Widget", "Gadget", "Widget", "Gizmo", "Widget"],
    "quantity": [10, 5, 8, 3, 12], "unit_price": [9.99, 19.99, 9.99, 14.99, 9.99],
    "quarter": "Q1",
})
sales_q2 = pd.DataFrame({
    "order_id": range(6, 12), "emp_id": [3, 4, 9, 9, 3, 2],
    "product": ["Widget", "Gadget", "Gizmo", "Widget", "Gadget", "Widget"],
    "quantity": [7, 6, 4, 9, 11, 2], "unit_price": [9.99, 19.99, 14.99, 9.99, 19.99, 9.99],
    "quarter": "Q2",
})
sales_q3 = pd.DataFrame({
    "order_id": range(12, 17), "emp_id": [4, 3, 9, 2, 3],
    "product": ["Gadget", "Widget", "Gizmo", "Widget", "Gizmo"],
    "quantity": [8, 15, 5, 6, 2], "unit_price": [19.99, 9.99, 14.99, 9.99, 14.99],
    "quarter": "Q3",
})

# ---- new tables ------------------------------------------------------------
sales_q4 = pd.DataFrame({
    "order_id": range(17, 23), "emp_id": [3, 4, 9, 2, 3, 4],
    "product": ["Widget", "Gizmo", "Gadget", "Widget", "Gadget", "Widget"],
    "quantity": [12, 6, 7, 5, 9, 14], "unit_price": [9.99, 14.99, 19.99, 9.99, 19.99, 9.99],
    "quarter": "Q4",
})

# Items sent back by customers (note: the key is order_id; not every order is returned)
returns = pd.DataFrame({
    "order_id": [3, 8, 14, 19],
    "returned_qty": [2, 1, 5, 3],
    "reason": ["Defective", "Wrong item", "Defective", "Changed mind"],
})

# Revenue targets. NOTE the column is called "department", not "dept_name"!
targets = pd.DataFrame({
    "department": ["Engineering"] * 4 + ["Sales"] * 4,
    "quarter": ["Q1", "Q2", "Q3", "Q4"] * 2,
    "target": [100, 100, 100, 100, 500, 550, 600, 650],
})

# Wide-format satisfaction ratings (one column per quarter)
ratings = pd.DataFrame({
    "name": ["Bob", "Chloe", "David", "Ines"],
    "Q1": [3.5, 4.2, 3.9, 4.0],
    "Q2": [3.8, 4.4, 3.7, 4.1],
    "Q3": [4.0, 4.1, 4.2, 3.8],
    "Q4": [4.3, 4.5, 4.0, 4.2],
})
print("pandas", pd.__version__)

## 1. Concatenation

**Exercise 1.1**
Concatenate `sales_q1` … `sales_q4` into one table called `year_sales` with a clean `0..n-1` index. Check that it has 22 rows, 6 columns, and that `order_id` is unique.

*Hint:* `ignore_index=True`; for uniqueness use `Series.is_unique`.

In [ ]:
# Solution
year_sales = pd.concat([sales_q1, sales_q2, sales_q3, sales_q4], ignore_index=True)
print(year_sales.shape)
assert year_sales.shape == (22, 6)
assert year_sales["order_id"].is_unique
year_sales.tail()

**Exercise 1.2**
Build the same concatenation with `keys=["Q1", "Q2", "Q3", "Q4"]` and `names=["src", "row"]` for the resulting index levels. Turn the `src` level into an ordinary column (without keeping `row`) and check that it always equals the existing `quarter` column.

*Hint:* `reset_index(level=...)` only moves the chosen level; a second `reset_index(drop=True)` discards the rest.

In [ ]:
# Solution
tagged = pd.concat(
    [sales_q1, sales_q2, sales_q3, sales_q4],
    keys=["Q1", "Q2", "Q3", "Q4"], names=["src", "row"],
)
tagged = tagged.reset_index(level="src").reset_index(drop=True)
assert (tagged["src"] == tagged["quarter"]).all()
tagged.head()

**Exercise 1.3**
Add a `discount` column to `sales_q4` (values `[0, .10, 0, .05, 0, .10]`) in a *copy* called `sales_q4_disc`. Concatenate `sales_q3` and `sales_q4_disc` once with `join="outer"` and once with `join="inner"`. How many `NaN` discounts are there in the outer version? Which columns survive in the inner version?

*Hint:* `DataFrame.assign` returns a copy, so `sales_q4` itself is untouched.

In [ ]:
# Solution
sales_q4_disc = sales_q4.assign(discount=[0.0, 0.10, 0.0, 0.05, 0.0, 0.10])
outer = pd.concat([sales_q3, sales_q4_disc], ignore_index=True)
inner = pd.concat([sales_q3, sales_q4_disc], join="inner", ignore_index=True)
print("NaN discounts (outer):", outer["discount"].isna().sum())   # 5: all the Q3 rows
print("columns (inner):", list(inner.columns))                     # no 'discount'

**Exercise 1.4** 
Try `pd.concat([sales_q1, sales_q2], verify_integrity=True)`. What happens and why? Then fix it so the concatenation succeeds *and* still verifies integrity.

*Hint:* `verify_integrity` checks the *index*, not the `order_id` column.

In [ ]:
# Solution
try:
    pd.concat([sales_q1, sales_q2], verify_integrity=True)
except ValueError as e:
    print("ValueError:", e)    # both frames have index labels 0..4 -> duplicates

ok = pd.concat([sales_q1, sales_q2], ignore_index=True, verify_integrity=True)
print(ok.shape)

**Exercise 1.5** 
A correction file arrives: order 17 was actually `quantity=13` (everything else identical). Append this one-row correction to `year_sales` so that the final table has **22 rows** and order 17 shows the corrected quantity. Why does a plain `.drop_duplicates()` *not* work here?

*Hint:* Think about *which columns* define a duplicate, and which copy you want to keep.

In [ ]:
# Solution
correction = pd.DataFrame([{"order_id": 17, "emp_id": 3, "product": "Widget",
                            "quantity": 13, "unit_price": 9.99, "quarter": "Q4"}])
combined = pd.concat([year_sales, correction], ignore_index=True)

print("plain drop_duplicates ->", len(combined.drop_duplicates()))              # still 23: rows differ in quantity
deduped = combined.drop_duplicates(subset="order_id", keep="last")
print("by order_id, keep last ->", len(deduped))                                # 22
assert len(deduped) == 22
deduped[deduped["order_id"] == 17]

## 2. Merge & join

**Checkpoint.** Parts 2–5 reuse `year_sales`. If you skipped exercise 1.1, run this cell:

In [ ]:
year_sales = pd.concat([sales_q1, sales_q2, sales_q3, sales_q4], ignore_index=True)

**Exercise 2.1**
Which employees have **never** made a sale? Use a `left` merge with `indicator=True` from `employees` to the *distinct* `emp_id`s found in `year_sales`, and keep only the `left_only` rows. Return just their names.

*Hint:* Drop duplicates on the right side first, otherwise a seller with several orders appears several times.

In [ ]:
# Solution
sellers = year_sales[["emp_id"]].drop_duplicates()
m = employees.merge(sellers, on="emp_id", how="left", indicator=True)
m.loc[m["_merge"] == "left_only", "name"].tolist()
# ['Alice', 'Emma', 'Farid', 'Grace', 'Hugo', 'Jack']

**Exercise 2.2**
Using `returns`, compute for every order `gross_revenue = quantity * unit_price` and `net_revenue = (quantity - returned_qty) * unit_price`. Orders without a return must count as `returned_qty = 0` (not `NaN`). Print total gross and total net revenue.

*Hint:* Which side must be `left` so that no order is lost? What does a left merge put in unmatched cells?

In [ ]:
# Solution
orders = year_sales.merge(returns, on="order_id", how="left")
orders["returned_qty"] = orders["returned_qty"].fillna(0).astype(int)
orders["gross_revenue"] = orders["quantity"] * orders["unit_price"]
orders["net_revenue"] = (orders["quantity"] - orders["returned_qty"]) * orders["unit_price"]

print(orders[["gross_revenue", "net_revenue"]].sum().round(2))
orders.loc[orders["returned_qty"] > 0, ["order_id", "quantity", "returned_qty", "net_revenue"]]

> **Teacher note (2.2):** Using `how="inner"` is the classic mistake: it would keep only the 4 returned orders.

**Exercise 2.3**
`targets` calls the department column `department`, while `departments` calls it `dept_name`. Merge them (keep **all** departments) using `left_on` / `right_on`, then remove the redundant column. Which departments have no target at all?

*Hint:* After the merge both key columns are present; one of them is redundant.

In [ ]:
# Solution
dt = departments.merge(targets, left_on="dept_name", right_on="department", how="left")
dt = dt.drop(columns="department")
print(len(dt), "rows")                                   # 10: Eng x4, Sales x4, Marketing, HR
dt.loc[dt["target"].isna(), "dept_name"].tolist()        # ['Marketing', 'HR']

**Exercise 2.4**
Reproduce `employees.merge(departments, on="dept_id", how="left")` using `.join()` instead (hint: one of the two frames must be indexed by `dept_id`, and `join` has an `on=` argument). Verify that `name`, `dept_name` and `location` are identical in both results.

*Hint:* `left.join(right, on=col)` matches the *column* `col` of `left` with the *index* of `right`.

In [ ]:
# Solution
a = employees.merge(departments, on="dept_id", how="left")
b = employees.join(departments.set_index("dept_id"), on="dept_id")
cols = ["name", "dept_name", "location"]
assert a[cols].equals(b[cols])
b[cols]

**Exercise 2.5**
Merges can silently inflate your data. (a) Try `employees.merge(departments, on="dept_id", validate="one_to_one")` and read the error. (b) Build `dept_dup` = `departments` with an extra duplicate copy of the Engineering row, merge `employees` with it (left join, no validation) and count the rows. (c) Show that `validate="many_to_one"` catches the problem.

*Hint:* `validate` accepts `one_to_one`, `one_to_many`, `many_to_one`, `many_to_many`.

In [ ]:
# Solution
# (a)
try:
    employees.merge(departments, on="dept_id", validate="one_to_one")
except pd.errors.MergeError as e:
    print("(a) MergeError:", e)

# (b)
dept_dup = pd.concat([departments, departments.iloc[[0]]], ignore_index=True)
inflated = employees.merge(dept_dup, on="dept_id", how="left")
print("(b) rows:", len(employees), "->", len(inflated))     # 10 -> 13 (Alice, Bob, Grace doubled)

# (c)
try:
    employees.merge(dept_dup, on="dept_id", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("(c) MergeError:", e)

**Exercise 2.6**
Build a table with **one row for every (employee name, quarter) combination** (10 × 4 = 40 rows) showing revenue, with `0` where the employee sold nothing. Use a `cross` merge to create the grid, then a `left` merge to attach revenue. How many cells are zero?

*Hint:* `how="cross"` takes no `on=`. Aggregate sales per (name, quarter) *before* the left merge.

In [ ]:
# Solution
rev = (
    year_sales.assign(revenue=lambda d: d["quantity"] * d["unit_price"])
    .merge(employees[["emp_id", "name"]], on="emp_id")
    .groupby(["name", "quarter"], as_index=False)["revenue"].sum()
)
grid = employees[["name"]].merge(
    pd.DataFrame({"quarter": ["Q1", "Q2", "Q3", "Q4"]}), how="cross"
)
full_grid = grid.merge(rev, on=["name", "quarter"], how="left").fillna({"revenue": 0})
assert len(full_grid) == 40
print("zero-revenue cells:", (full_grid["revenue"] == 0).sum())
full_grid.head(8)

## 3. GroupBy

**Checkpoint.** This cell builds the enriched sales table used below (re-run it if you changed `year_sales`):

In [ ]:
sales_full = (
    year_sales
    .merge(employees, on="emp_id", how="left")
    .merge(departments, on="dept_id", how="left")
    .assign(revenue=lambda d: d["quantity"] * d["unit_price"])
)
sales_full.head()

**Exercise 3.1**
For each product compute: number of orders, total quantity, average unit price and total revenue, using *named aggregation*. Sort by total revenue (descending) and round to 2 decimals.

*Hint:* `.agg(new_name=("column", "function"))`

In [ ]:
# Solution
(sales_full.groupby("product")
    .agg(n_orders=("order_id", "count"),
         total_quantity=("quantity", "sum"),
         avg_unit_price=("unit_price", "mean"),
         total_revenue=("revenue", "sum"))
    .sort_values("total_revenue", ascending=False)
    .round(2))

**Exercise 3.2**
Who is the top seller (by total revenue) in **each quarter**? Return one row per quarter with the employee name and their revenue.

*Hint:* Group twice: first by (quarter, name) to sum, then by quarter to find the max.

In [ ]:
# Solution
rev = sales_full.groupby(["quarter", "name"])["revenue"].sum()
top_idx = rev.groupby(level="quarter").idxmax()           # (quarter, name) tuples
top = rev.loc[top_idx.values].reset_index()
top
# Alternative: rev.reset_index().sort_values("revenue", ascending=False).groupby("quarter").head(1)

**Exercise 3.3**
On the `employees` table add two columns: `salary_rank_in_dept` (1 = best paid in the department) and `salary_vs_dept_avg` (salary minus the department's mean salary). What value do Hugo and Jack get for the rank, and why?

*Hint:* `groupby(...)[col].rank(...)` and `.transform("mean")` both return a result aligned with the original rows.

In [ ]:
# Solution
emp = employees.copy()
emp["salary_rank_in_dept"] = emp.groupby("dept_id")["salary"].rank(ascending=False, method="min")
emp["salary_vs_dept_avg"] = emp["salary"] - emp.groupby("dept_id")["salary"].transform("mean")
print(emp[["name", "dept_id", "salary", "salary_rank_in_dept", "salary_vs_dept_avg"]])
# Hugo (dept 40) and Jack (dept 99) are alone in their group -> rank 1, difference 0.

**Exercise 3.4**
Add a `cum_revenue` column to a copy of `sales_full`: the running total of each employee's revenue, in order of `order_id`. Show Chloe's rows.

*Hint:* Sort *before* the cumulative sum. `groupby(...).cumsum()`.

In [ ]:
# Solution
cum = sales_full.sort_values("order_id").copy()
cum["cum_revenue"] = cum.groupby("name")["revenue"].cumsum()
cum.loc[cum["name"] == "Chloe", ["order_id", "quarter", "revenue", "cum_revenue"]]

**Exercise 3.5**
Using `filter`, keep only the rows of `sales_full` that belong to departments with **at least 5 orders**. Which departments remain?

*Hint:* Inside `filter` the lambda receives each group's sub-DataFrame and must return a single boolean.

In [ ]:
# Solution
busy = sales_full.groupby("dept_name").filter(lambda g: len(g) >= 5)
busy["dept_name"].unique()      # ['Sales'] (Engineering has only 3 orders)

**Exercise 3.6**
Compute each employee's tenure in years at `2025-01-01`, bucket it with `pd.cut` into `0-2y`, `2-4y`, `4+y` (bins `[0, 2, 4, 10]`), and report, per bucket, the number of employees and their average salary.

*Hint:* Subtracting two datetimes gives a timedelta; `.dt.days` extracts the number of days. Use `observed=True` to avoid empty categories.

In [ ]:
# Solution
ref = pd.Timestamp("2025-01-01")
emp = employees.assign(tenure_years=(ref - employees["hire_date"]).dt.days / 365.25)
emp["bucket"] = pd.cut(emp["tenure_years"], bins=[0, 2, 4, 10], labels=["0-2y", "2-4y", "4+y"])
emp.groupby("bucket", observed=True).agg(n=("emp_id", "count"), avg_salary=("salary", "mean"))

## 4. Reshaping & pivot tables

**Exercise 4.1**
Build a `pivot_table` of total revenue with employee `name` as rows and `quarter` as columns, missing values as 0, with a `Total` row and column. Round to 2 decimals.

*Hint:* `margins=True, margins_name="Total"`

In [ ]:
# Solution
sales_full.pivot_table(index="name", columns="quarter", values="revenue",
                       aggfunc="sum", fill_value=0,
                       margins=True, margins_name="Total").round(2)

**Exercise 4.2**
From a product × quarter revenue pivot, compute the **quarter-over-quarter growth** (e.g. Q2 vs Q1) for each product, as a table rounded to 3 decimals. Which product has the largest growth between Q3 and Q4?

*Hint:* `DataFrame.pct_change(axis=1)` works across columns.

In [ ]:
# Solution
rev_pq = sales_full.pivot_table(index="product", columns="quarter", values="revenue",
                                aggfunc="sum", fill_value=0)
growth = rev_pq.pct_change(axis=1).round(3)          # first column (Q1) is NaN: nothing to compare with
print(growth)
print("Best Q3->Q4 growth:", growth["Q4"].idxmax())

**Exercise 4.3**
`ratings` is in wide format. (a) `melt` it into long format with columns `name, quarter, rating`. (b) Compute the average rating per quarter and the employee with the best average rating. (c) `pivot` the long table back to the original wide shape.

*Hint:* `id_vars` are the columns you do NOT want to unpivot.

In [ ]:
# Solution
long_r = ratings.melt(id_vars="name", var_name="quarter", value_name="rating")
print(long_r.groupby("quarter")["rating"].mean().round(3))
print("Best average:", long_r.groupby("name")["rating"].mean().idxmax())

wide_again = long_r.pivot(index="name", columns="quarter", values="rating")
wide_again

**Exercise 4.4**
Compute total `quantity` per (`name`, `product`) as a Series, then `unstack` the product level with zeros for missing combinations. Check that the grand total of the wide table equals the total of the Series. Then `stack` back and recover *only* the combinations that really occurred.

*Hint:* `unstack(level, fill_value=0)`. Stacking a table filled with 0 brings the zeros back as real rows.

In [ ]:
# Solution
qty = sales_full.groupby(["name", "product"])["quantity"].sum()
wide = qty.unstack("product", fill_value=0)
assert wide.to_numpy().sum() == qty.sum()
print(wide)

back = wide.stack()
real_only = back[back > 0]
assert len(real_only) == len(qty)

**Exercise 4.5**
Why does `sales_full.pivot(index="name", columns="quarter", values="revenue")` fail? Show the error, then produce the intended table in **two different ways** (`groupby` + `unstack`, and `pivot_table`).

*Hint:* Chloe has several orders in the same quarter.

In [ ]:
# Solution
try:
    sales_full.pivot(index="name", columns="quarter", values="revenue")
except ValueError as e:
    print("ValueError:", e)      # duplicate (name, quarter) pairs: pivot never aggregates

v1 = sales_full.groupby(["name", "quarter"])["revenue"].sum().unstack("quarter")
v2 = sales_full.pivot_table(index="name", columns="quarter", values="revenue", aggfunc="sum")
assert v1.round(6).equals(v2.round(6).rename_axis(columns="quarter"))
v1

> **Teacher note (4.5):** `pivot_table` defaults to `aggfunc="mean"`, so forgetting `aggfunc="sum"` gives wrong numbers silently.

**Exercise 4.6**
With `pd.crosstab`, show for each department the **share of quantity** sold by product (each row sums to 1). Then check that the rows do sum to 1.

*Hint:* `normalize="index"` divides each row by its total; with `values=` you must also pass `aggfunc`.

In [ ]:
# Solution
share = pd.crosstab(sales_full["dept_name"], sales_full["product"],
                    values=sales_full["quantity"], aggfunc="sum", normalize="index")
assert np.allclose(share.sum(axis=1), 1)
share.round(3)